# 05 - Recursive Multi-Step Demand Forecasting

### Overview
- Load trained LightGBM model and historical sales records.
- Execute multi-day recursive forecasting across the test period.
- Iteratively generate lag and rolling window features from predictions.
- Save final demand forecast predictions to `outputs/predictions/demand_forecast.csv`.


In [43]:
import os
import pandas as pd
import numpy as np
import joblib

base_dir = ".." if os.path.exists("../data") else "."

# Load trained model
model = joblib.load(
    f"{base_dir}/models/lightgbm_inventory_model.pkl"
)

# Load original historical sales
historical = pd.read_csv(f"{base_dir}/data/raw/train.csv")

# Load feature-engineered datasets
train_features = pd.read_csv(
    f"{base_dir}/data/processed/train_features.csv"
)

test = pd.read_csv(
    f"{base_dir}/data/processed/test_features.csv"
)

# Convert dates
for data in [historical, train_features, test]:
    data["date"] = pd.to_datetime(data["date"])

print("Historical data:", historical.shape)
print("Test data:", test.shape)


Historical data: (3000888, 6)
Test data: (28512, 22)


In [44]:
# Basic checks before forecasting
required_history_columns = {"date", "store_nbr", "family", "sales"}
required_test_columns = {"id", "date", "store_nbr", "family"}

assert required_history_columns.issubset(historical.columns), (
    f"Historical data is missing: {required_history_columns - set(historical.columns)}"
)
assert required_test_columns.issubset(test.columns), (
    f"Test data is missing: {required_test_columns - set(test.columns)}"
)

print("Historical date range:", historical["date"].min(), "to", historical["date"].max())
print("Forecast date range:", test["date"].min(), "to", test["date"].max())
print("Historical rows:", len(historical))
print("Forecast rows:", len(test))


Historical date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Forecast date range: 2017-08-16 00:00:00 to 2017-08-31 00:00:00
Historical rows: 3000888
Forecast rows: 28512


In [45]:
historical = historical.sort_values("date")

history = {}

for (store, family), group in historical.groupby(
    ["store_nbr", "family"]
):
    history[(store, family)] = dict(
        zip(group["date"], group["sales"])
    )

print("Store-product combinations:", len(history))

Store-product combinations: 1782


In [46]:
def generate_lag_features(day_data, history):
    day_data = day_data.copy()

    lag_7 = []
    lag_14 = []
    lag_28 = []
    rolling_7 = []
    rolling_28 = []

    for row in day_data.itertuples(index=False):
        key = (row.store_nbr, row.family)
        past_sales = history.get(key, {})
        current_date = pd.Timestamp(row.date)

        # Exact values from 7, 14, and 28 days earlier.
        lag_7.append(
            past_sales.get(current_date - pd.Timedelta(days=7), np.nan)
        )
        lag_14.append(
            past_sales.get(current_date - pd.Timedelta(days=14), np.nan)
        )
        lag_28.append(
            past_sales.get(current_date - pd.Timedelta(days=28), np.nan)
        )

        # Mean of the available sales in the previous 7 and 28 days.
        previous_7 = [
            past_sales.get(current_date - pd.Timedelta(days=i), np.nan)
            for i in range(1, 8)
        ]
        previous_28 = [
            past_sales.get(current_date - pd.Timedelta(days=i), np.nan)
            for i in range(1, 29)
        ]

        rolling_7.append(
            np.nan if np.isnan(previous_7).all()
            else float(np.nanmean(previous_7))
        )
        rolling_28.append(
            np.nan if np.isnan(previous_28).all()
            else float(np.nanmean(previous_28))
        )

    day_data["lag_7"] = lag_7
    day_data["lag_14"] = lag_14
    day_data["lag_28"] = lag_28
    day_data["rolling_mean_7"] = rolling_7
    day_data["rolling_mean_28"] = rolling_28

    return day_data


In [47]:
# Exact features expected by the trained LightGBM model
feature_columns = model.feature_name_

# Check that the static model features exist in the test data.
# The five lag/rolling features are intentionally created below.
lag_columns = [
    "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_28"
]
missing_static = [
    col for col in feature_columns
    if col not in test.columns and col not in lag_columns
]

if missing_static:
    raise ValueError(
        "These model features are missing from test_features.csv: "
        f"{missing_static}. Rebuild test_features.csv using the same "
        "feature-engineering steps used during model training."
    )

# The saved LightGBM model contains 4 categorical definitions.
# In this project these correspond to the original categorical columns:
# family, city, state, and type.
#
# Do not infer these from CSV dtypes: CSV loading can change category
# columns back into strings or numbers.
categorical_columns = ["family", "city", "state", "type"]

missing_categorical = [
    col for col in categorical_columns
    if col not in feature_columns or col not in test.columns
]
if missing_categorical:
    raise ValueError(
        "Expected categorical columns are missing: "
        f"{missing_categorical}. Check the feature-engineering columns."
    )

saved_categories = getattr(model.booster_, "pandas_categorical", None)
if not saved_categories or len(saved_categories) != len(categorical_columns):
    raise ValueError(
        "The model does not contain the expected four category definitions. "
        f"Saved definitions: {0 if saved_categories is None else len(saved_categories)}. "
        "The categorical columns used during model training must be checked."
    )

# Pair each known categorical feature with its saved category levels.
category_mappings = dict(zip(categorical_columns, saved_categories))

# Check that each saved category definition is compatible with its column.
for col, categories in category_mappings.items():
    observed = set(test[col].dropna().astype(str).unique())
    allowed = set(pd.Series(categories).dropna().astype(str).unique())
    unseen = observed - allowed
    if unseen:
        print(
            f"Warning: {len(unseen)} unseen value(s) in {col}; "
            "they will be treated as missing by LightGBM."
        )

print("Model features:", feature_columns)
print("Categorical columns:", categorical_columns)
print("Saved category counts:", {
    col: len(categories)
    for col, categories in category_mappings.items()
})

# Forecast one date at a time so each day's predictions become
# historical values for the following dates.
all_predictions = []
future_dates = sorted(pd.to_datetime(test["date"].unique()))

for current_date in future_dates:
    current_date = pd.Timestamp(current_date)
    print("Forecasting:", current_date.date())

    day_data = test.loc[test["date"] == current_date].copy()

    # Generate the five historical sales features.
    day_data = generate_lag_features(day_data, history)

    # Apply the exact category levels stored in the trained model.
    for col, categories in category_mappings.items():
        # Use the exact saved category labels and order from model training.
        day_data[col] = pd.Categorical(
            day_data[col].astype(str),
            categories=[str(value) for value in categories]
        )

    # Select features in exactly the order used during training.
    X_future = day_data.loc[:, feature_columns]

    if X_future.columns.tolist() != feature_columns:
        raise ValueError("Model feature names or order do not match.")

    # Predict demand.
    predictions = model.predict(X_future)
    predictions = np.maximum(np.asarray(predictions), 0)

    day_data["predicted_sales"] = predictions

    all_predictions.append(
        day_data[
            ["id", "date", "store_nbr", "family", "predicted_sales"]
        ].copy()
    )

    # Add today's predictions to history for recursive forecasting.
    for row, prediction in zip(day_data.itertuples(index=False), predictions):
        key = (row.store_nbr, row.family)
        if key not in history:
            history[key] = {}
        history[key][current_date] = float(prediction)

# Combine all daily forecasts.
if not all_predictions:
    raise ValueError("No future dates were found in test_features.csv.")

forecast = pd.concat(all_predictions, ignore_index=True)

print("\\nForecasting completed!")
print("Forecast shape:", forecast.shape)
display(forecast.head(10))


Model features: ['store_nbr', 'family', 'onpromotion', 'year', 'month', 'day', 'week_name', 'year_name', 'week_of_year', 'quarter', 'is_weekend', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'has_promotion', 'promotion_intensity', 'is_national_holiday', 'holiday_count', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']
Categorical columns: ['family', 'city', 'state', 'type']
Saved category counts: {'family': 33, 'city': 22, 'state': 16, 'type': 5}
Forecasting: 2017-08-16
Forecasting: 2017-08-17
Forecasting: 2017-08-18
Forecasting: 2017-08-19
Forecasting: 2017-08-20
Forecasting: 2017-08-21
Forecasting: 2017-08-22
Forecasting: 2017-08-23
Forecasting: 2017-08-24
Forecasting: 2017-08-25
Forecasting: 2017-08-26
Forecasting: 2017-08-27
Forecasting: 2017-08-28
Forecasting: 2017-08-29
Forecasting: 2017-08-30
Forecasting: 2017-08-31
\nForecasting completed!
Forecast shape: (28512, 5)


,id,date,store_nbr,family,predicted_sales
0,3000888,2017-08-16,1,AUTOMOTIVE,26.908013
1,3000889,2017-08-16,1,BABY CARE,26.908013
2,3000890,2017-08-16,1,BEAUTY,26.908013
3,3000891,2017-08-16,1,BEVERAGES,2042.168486
4,3000892,2017-08-16,1,BOOKS,26.908013
5,3000893,2017-08-16,1,BREAD/BAKERY,358.931132
6,3000894,2017-08-16,1,CELEBRATION,26.908013
7,3000895,2017-08-16,1,CLEANING,603.652527
8,3000896,2017-08-16,1,DAIRY,712.213347
9,3000897,2017-08-16,1,DELI,131.623944


In [48]:
import os
import pandas as pd

output_dir = f"{base_dir}/outputs/predictions"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/demand_forecast.csv"

forecast.to_csv(output_path, index=False)

print("Demand forecast saved successfully!")
print("File location:", os.path.abspath(output_path))


Demand forecast saved successfully!
File location: d:\Store Prdiction\outputs\predictions\demand_forecast.csv


In [49]:
pd.read_csv(f"{base_dir}/outputs/predictions/demand_forecast.csv")


,id,date,store_nbr,family,predicted_sales
0,3000888,2017-08-16,1,AUTOMOTIVE,26.908013
1,3000889,2017-08-16,1,BABY CARE,26.908013
2,3000890,2017-08-16,1,BEAUTY,26.908013
3,3000891,2017-08-16,1,BEVERAGES,2042.168486
4,3000892,2017-08-16,1,BOOKS,26.908013
...,...,...,...,...,...
28507,3029395,2017-08-31,9,POULTRY,376.650041
28508,3029396,2017-08-31,9,PREPARED FOODS,145.343447
28509,3029397,2017-08-31,9,PRODUCE,1215.331111
28510,3029398,2017-08-31,9,SCHOOL AND OFFICE SUPPLIES,146.653416
